# Visualization 1

In [1]:
import pandas as pd
import altair as alt

pd.Series.iteritems = pd.Series.items   # Fix for compatibility issues between Pandas and Altair
alt.data_transformers.enable("json")   # let Altair handle the large dataset (lifts the 5000-row limit)

DataTransformerRegistry.enable('json')

In [2]:
data_dir="../data/"

In [3]:
names = pd.read_csv(f"{data_dir}dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

# Clean and parse the year values to integers
names["annais"] = pd.to_numeric(names["annais"], errors="coerce")
names.dropna(subset=["annais"], inplace=True)
names["annais"] = names["annais"].astype(int)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2542735,2,GISELLE,1946,13,5
3276141,2,NADEGE,1984,13,21
2749997,2,KATE,2003,75,3
2188186,2,CLAUDIE,1950,30,11
3397980,2,PAULINE,1909,20,49


Let's explore the data

In [4]:
print("Number of distinct names :", names.preusuel.nunique())
print("Number of departments    :", names.dpt.nunique())
print(f"Years from {names.annais.min()}  to  {names.annais.max()} ")

Number of distinct names : 15270
Number of departments    : 99
Years from 1900  to  2020 


Let's display the most popular names in France

In [5]:
popularity = names.groupby("preusuel")["nombre"].sum()
popularity.sort_values(ascending=False).head(10)

preusuel
MARIE       2256072
JEAN        1913130
PIERRE       891794
MICHEL       818025
ANDRÉ        709633
JEANNE       556903
PHILIPPE     535355
LOUIS        523576
RENÉ         514560
ALAIN        504106
Name: nombre, dtype: int64

Let's build our visualization 1

In [8]:
# Set to 10 for decades
# 5 for a 5 year step
# 1 for individual years
TIME_STEP = 10

# Group the continuous years into fixed period blocks based TIME_STEP
names["period"] = (names["annais"] // TIME_STEP) * TIME_STEP

# Extract the 10 top popular names

top_10_names = names.groupby("preusuel")["nombre"].sum().nlargest(10).index.tolist()

# Aggregate totals to a national scale for the heatmap matrix
df_heatmap = names[names.preusuel.isin(top_10_names)].groupby(["preusuel", "period"], as_index=False)["nombre"].sum()

# --------------------------------------------------
# Build Visualization 1
# --------------------------------------------------

# --- Sliders for dynamic time interval boundaries ---
start_slider = alt.binding_range(min=1900, max=2020, step=TIME_STEP, name="Start Year: ")
start_select = alt.selection_point(
    fields=['period'], 
    bind=start_slider, 
    value=[{'period': 1900}], # Syntax compliant for modern Altair schema rules
    name="start_bound"
)

end_slider = alt.binding_range(min=1900, max=2020, step=TIME_STEP, name="End Year: ")
end_select = alt.selection_point(
    fields=['period'], 
    bind=end_slider, 
    value=[{'period': 2020}], 
    name="end_bound"
)

# --- Dropdown selection list ---
name_dropdown = alt.binding_select(
    options=[None] + top_10_names, 
    labels=["All Top 10 Names"] + top_10_names, 
    name="Focus Name: "
)

name_select = alt.selection_point(
    fields=["preusuel"], 
    bind=name_dropdown, 
    name="name_filter"
)

# --- Heatmap specifics ---
heatmap_chart = alt.Chart(df_heatmap).mark_rect().encode(
    x=alt.X(
        "period:O", 
        title="Time Period / Year",
        axis=alt.Axis(labelAngle=-45, labelAlign='right')
    ),
    y=alt.Y(
        "preusuel:N", 
        title="Baby Name",
        sort=alt.EncodingSortField(field="nombre", op="sum", order="descending")
    ),
    color=alt.Color(
        "nombre:Q", 
        title="Total Births", 
        scale=alt.Scale(scheme="yellowgreenblue")
    ),
    tooltip=[
        alt.Tooltip("preusuel:N", title="Name"),
        alt.Tooltip("period:O", title="Period Starts"),
        alt.Tooltip("nombre:Q", title="Total Registered Births", format=",")
    ]
).properties(
    width=750,
    height=350,
    title=f"National Baby Name Intensity Grid ({TIME_STEP}-Year Time Intervals)"
).add_params(       # FIX: In modern Altair, use add_params() instead of add_selection()
    start_select, 
    end_select, 
    name_select
).transform_filter(
    alt.datum.period >= start_select.period
).transform_filter(
    alt.datum.period <= end_select.period
).transform_filter(
    name_select
)

heatmap_chart

alt.Chart(...)